# Keras 3 + MLflow — control hub

Everything below drives three containers. Nothing trains on this kernel.

| container | role | lifecycle |
|---|---|---|
| `mlflow-server` | tracking + model registry, sqlite backend | long running, port 5000 |
| `trainer` | Keras 3 (torch backend), one run per launch | launched per run, exits |
| `api-serve` | FastAPI + uvicorn, serves `@champion` | you start it, port 8000 |

The flow: **train a few runs → compare them → promote one to `@champion` → serve it → watch
the predictions**. Models live only in the MLflow registry; the host keeps `./mlruns` (the
store), `./data` (cached MNIST) and `./inference` (every prediction, as JSONL).

In [3]:
import json, os, time
from pathlib import Path

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
os.chdir(PROJECT)     # every `!docker ...` line below runs from the project root
print("project:", PROJECT)

!docker compose version

project: /home/biesdorf/MLOps/keras3-with-mlflow
Docker Compose version v5.4.0


## 1. Build the images

First build pulls CPU-only torch, so expect a few minutes. Later builds are cached.

In [4]:
!docker compose --profile tools --profile serve build

[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.1s (1/2)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 1.59kB                                           0.0s
[+] Building 0.3s (4/5)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 1.59kB                                           0.0s
 => [trainer internal] load build definition from Dockerfile               0.0s
 => => transferring dockerfile: 393B                                       0.0s
 => [api-serve internal] load build definition from Dockerfile             0.0s
 => => transferring dockerfile: 370B                                       0.0s
 => [mlflow-server internal] load bui

## 2. Start (or restart) the MLflow server

Reusable and standalone: it knows nothing about Keras. Open <http://localhost:5000> once it is healthy.

The cell force-recreates the container, so **re-run it any time you want a clean restart** —
after changing `docker-compose.yml`, or when the server gets wedged. The tracking data lives in
`./mlruns` on the host, so a restart never loses runs, models, or aliases.

In [5]:
import urllib.request

# --force-recreate always tears the container down and builds a fresh one, so
# re-running this cell is a real restart instead of a no-op "Container Running".
# Safe to repeat: all state lives in ./mlruns on the host, not in the container.
!docker compose up -d --force-recreate mlflow-server

for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:5000/health", timeout=2).read()
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("mlflow-server did not come up")

print("UI: http://localhost:5000")

[+] up 0/1
 ⠋ Container mlflow-server Recreate                                         0.0s
[+] up 0/1
 ⠙ Container mlflow-server Recreate                                         0.1s
[+] up 0/1
 ⠹ Container mlflow-server Recreate                                         0.2s
[+] up 0/1
 ⠸ Container mlflow-server Recreate                                         0.3s
[+] up 0/1
 ⠼ Container mlflow-server Recreate                                         0.4s
[+] up 0/1
 ⠴ Container mlflow-server Recreate                                         0.5s
[+] up 0/1
 ⠦ Container mlflow-server Recreate                                         0.6s
[+] up 0/1
 ⠧ Container mlflow-server Recreate                                         0.7s
[+] up 0/1
 ⠇ Container mlflow-server Recreate                                         0.8s
[+] up 0/1
 ⠏ Container mlflow-server Recreate                                         0.9s
[+] up 0/1
 ⠋ Container mlflow-server Recreate                                  

## 3. Train — three runs, one cell at a time

Each cell launches the **trainer container** with its hyperparameters as `-e VAR=value`. The
container trains one model, logs one MLflow run, registers a model version, and exits. Nothing
is baked into the image, so changing a number and re-running the cell is the whole edit loop —
no rebuild.

Three runs is enough to make the comparison in the MLflow UI mean something: a baseline, a
better architecture, and one run you expect to lose.

In [6]:
def train(name, **params):
    """Launch the trainer container for ONE run.

    Every keyword becomes an -e VAR=value on `docker compose run`, so the whole
    command is visible in the cell output. Knobs: kind (mlp/cnn), size, dropout,
    opt (adam/sgd), lr, epochs, batch_size, train_subset.
    """
    env = " ".join(f"-e {k.upper()}={v}"
                   for k, v in dict(run_name=name, **params).items())
    !docker compose --profile tools run --rm {env} trainer


print(train.__doc__.splitlines()[0])

Launch the trainer container for ONE run.


### Run 1 — the baseline

One 128-unit hidden layer. Note the accuracy and the seconds; the other runs are judged
against these two numbers.

In [7]:
train("mlp-128", kind="mlp", size=128, dropout=0.2, opt="adam", lr="1e-3")

[+] create 1/1
 ✔ Container mlflow-server Running                                          0.0s
[+] start 1/1
 ✔ Container mlflow-server Running                                          0.0s
[+]  1/1
 ✔ Container mlflow-server Running                                          0.0s
Container mlflow-server Waiting 
Container mlflow-server Healthy 
Container keras-mlflow-mnist-trainer-run-578ba9a2da48 Creating 
Container keras-mlflow-mnist-trainer-run-578ba9a2da48 Created 
[keras] 3.15.1 backend=torch  mlflow=3.15.1
[config] {'name': 'mlp-128', 'kind': 'mlp', 'size': 128, 'dropout': 0.2, 'opt': 'adam', 'lr': 0.001} epochs=3 batch_size=128 subset=20000
[data] loaded /data/mnist.npz
[data] train=(20000, 28, 28, 1) test=(10000, 28, 28, 1)
Epoch 1/3
141/141 - 1s - 6ms/step - accuracy: 0.8101 - loss: 0.6682 - val_accuracy: 0.9275 - val_loss: 0.2798
Epoch 2/3
141/141 - 1s - 6ms/step - accuracy: 0.9116 - loss: 0.3115 - val_accuracy: 0.9425 - val_loss: 0.2225
Epoch 3/3
141/141 - 1s - 7ms/step - ac

### Run 2 — a different architecture

A conv block instead of a dense layer, and 5 epochs instead of the default 3. Fewer parameters
than run 1, a clearly better score — this is the run that ends up as `@champion`.

In [ ]:
train("cnn-32", kind="cnn", size=32, dropout=0.25, opt="adam", lr="1e-3", epochs=5)

### Run 3 — a deliberately bad control

No dropout, SGD at a learning rate that is too large. A comparison needs at least one run you
expect to lose, otherwise you cannot tell a good score from an easy dataset.

In [ ]:
train("cnn-16-sgd", kind="cnn", size=16, dropout=0.0, opt="sgd", lr="3e-2")

## 4. Compare the runs, then promote the winner to `@champion`

Compare the three runs in the MLflow UI: <http://localhost:5000> → experiment `mnist-keras` →
sort by `test_accuracy`, or tick two runs and hit **Compare**.

Then the promotion. An **alias** is a movable pointer on a registered model. `api-serve`
resolves `models:/mnist-keras@champion` at startup, so promotion is one API call plus a
restart. The cell picks the best run by `test_accuracy` — to promote a different one, pass its
name as `RUN_NAME`.

In [ ]:
import mlflow
from mlflow import MlflowClient

RUN_NAME = None      # None = whichever run scored best; or e.g. "cnn-32"

mlflow.set_tracking_uri("http://localhost:5000")
runs = mlflow.search_runs(
    experiment_names=["mnist-keras"],
    filter_string=f"tags.mlflow.runName = '{RUN_NAME}'" if RUN_NAME else "",
    order_by=["metrics.test_accuracy DESC"], max_results=1)

if runs.empty:
    raise RuntimeError("no matching run — train at least one model first")

best = runs.iloc[0]
client = MlflowClient()

# find the registered version produced by that run
version = next(v for v in client.search_model_versions("name='mnist-keras'")
               if v.run_id == best["run_id"])

client.set_registered_model_alias("mnist-keras", "champion", version.version)
client.set_model_version_tag("mnist-keras", version.version, "promoted_by", "hub-notebook")

print(f"champion -> version {version.version} "
      f"({best['tags.mlflow.runName']}, acc={best['metrics.test_accuracy']:.4f})")

## 5. Start the serving container

It resolves the alias once at startup, and has never seen `train.py` — the registry is the only
thing the two containers share. Re-run this cell after promoting a different version, or
`POST /reload` to swap without a restart.

In [ ]:
import requests

!docker compose --profile serve up -d --force-recreate api-serve

for _ in range(60):
    try:
        health = requests.get("http://localhost:8000/health", timeout=2).json()
        break
    except Exception:
        time.sleep(2)
else:
    !docker compose logs --tail 50 api-serve
    raise RuntimeError("api-serve did not come up")

print(json.dumps(health, indent=2))

## 6. Example inference

Two ways in, and both hit the same champion:

- `GET /predict/sample/{i}` — the container looks the digit up in its own cached copy of the
  MNIST test set. Handy for `curl`. The cell asks for the first ten.
- `POST /predict` — you send 784 raw pixels. The cell draws three of those ten digits, picked
  at random, and posts them, so you can check the prediction against what you can see.

Both indexes refer to the same test set, so `sample 3` and the third drawing are the same image.

```bash
curl localhost:8000/health
curl localhost:8000/predict/sample/7
```

In [17]:
import numpy as np

# a) ten digits, picked by index — the container reads them from its own cache
for i in range(10):
    r = requests.get(f"http://localhost:8000/predict/sample/{i}").json()
    hit = "  " if r["prediction"] == r["label"] else "<-- miss"
    print(f"sample {i}: true={r['label']} pred={r['prediction']} "
          f"conf={r['confidence']} {hit}")

# b) three of those ten at random, sent as raw pixels straight from the npz
d = np.load(PROJECT / "data" / "mnist.npz")
picks = sorted(np.random.choice(10, size=3, replace=False))

art = [["".join(" .:-=+*#%@"[p // 26] for p in row) for row in d["x_test"][i]]
       for i in picks]
print("\n" + "  ".join(f"sample {i}".ljust(28) for i in picks))
for line in zip(*art):
    print("  ".join(line))

for i in picks:
    img, truth = d["x_test"][i], int(d["y_test"][i])
    r = requests.post("http://localhost:8000/predict",
                      json={"pixels": img.flatten().tolist(), "label": truth}).json()
    print(f"sample {i}: true={truth}  pred={r['prediction']}  conf={r['confidence']}")

sample 0: true=7 pred=7 conf=1.0   
sample 1: true=2 pred=2 conf=0.9996   
sample 2: true=1 pred=1 conf=0.9977   
sample 3: true=0 pred=0 conf=1.0   
sample 4: true=4 pred=4 conf=0.9988   
sample 5: true=1 pred=1 conf=0.9981   
sample 6: true=4 pred=4 conf=0.9989   
sample 7: true=9 pred=9 conf=0.8961   
sample 8: true=5 pred=5 conf=0.9099   
sample 9: true=9 pred=9 conf=0.9938   

sample 1                      sample 4                      sample 6                    
                                                                                        
                                                                                        
                                                                                        
          ==*@@+-                                                                       
         *@@@@@@%.                                                                      
        *@@@%+*@@=                      .%       :.                   #+.        

## 7. Inference log

Every prediction was appended to `./inference/predictions.jsonl` on the host, so you never
have to shell into the container to see what a running model is doing. Note the `version`
column: that is the champion you promoted in section 4.

In [18]:
import pandas as pd

log = PROJECT / "inference" / "predictions.jsonl"
rows = [json.loads(l) for l in log.read_text().splitlines()]
pd.DataFrame(rows)[["ts", "model", "version", "source", "label", "prediction", "confidence"]].tail(10)

,ts,model,version,source,label,prediction,confidence
51,2026-08-17T10:26:45.916307+00:00,mnist-keras,20,mnist_test[3],0,0,1.0000
52,2026-08-17T10:26:45.958397+00:00,mnist-keras,20,mnist_test[4],4,4,0.9988
53,2026-08-17T10:26:46.002878+00:00,mnist-keras,20,mnist_test[5],1,1,0.9981
54,2026-08-17T10:26:46.047013+00:00,mnist-keras,20,mnist_test[6],4,4,0.9989
55,2026-08-17T10:26:46.091207+00:00,mnist-keras,20,mnist_test[7],9,9,0.8961
56,2026-08-17T10:26:46.133523+00:00,mnist-keras,20,mnist_test[8],5,5,0.9099
57,2026-08-17T10:26:46.175441+00:00,mnist-keras,20,mnist_test[9],9,9,0.9938
58,2026-08-17T10:26:46.255033+00:00,mnist-keras,20,request,2,2,0.9996
59,2026-08-17T10:26:46.283368+00:00,mnist-keras,20,request,4,4,0.9988
60,2026-08-17T10:26:46.310198+00:00,mnist-keras,20,request,4,4,0.9989


## 8. Teardown

No cell here on purpose — the notebook should never stop the stack out from under you on a
**Run All**. When you are done exploring the MLflow UI, run this in a terminal, from the
project root:

```bash
docker compose --profile tools --profile serve down
```

Nothing is lost. Runs, models and the `@champion` alias live in `./mlruns` on the host, not in
the containers, and `./inference` and `./data` survive too. Restart from section 2.